In [1]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression

# Page Configuration
st.set_page_config(
    page_title="Student Performance Predictor",
    page_icon="🎓",
    layout="wide",
    initial_sidebar_state="expanded"
)

# Custom CSS for Professional Enterprise UI Styling
st.markdown("""
    <style>
    .main-header {
        font-size: 2.5rem;
        color: #1E3A8A;
        font-weight: 800;
        margin-bottom: 0px;
        letter-spacing: -0.5px;
    }
    .sub-header {
        font-size: 1.15rem;
        color: #4B5563;
        margin-bottom: 25px;
        font-weight: 400;
    }
    .card-container {
        background-color: #FFFFFF;
        padding: 20px;
        border-radius: 12px;
        box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.1), 0 2px 4px -1px rgba(0, 0, 0, 0.06);
        border: 1px solid #E5E7EB;
        margin-bottom: 20px;
    }
    .stButton>button {
        background-color: #2563EB;
        color: white;
        font-weight: 600;
        border-radius: 8px;
        padding: 0.6rem 1rem;
        border: none;
        box-shadow: 0 2px 4px rgba(37, 99, 235, 0.3);
        transition: all 0.3s ease;
    }
    .stButton>button:hover {
        background-color: #1D4ED8;
        box-shadow: 0 4px 8px rgba(37, 99, 235, 0.4);
    }
    </style>
""", unsafe_allow_html=True)

CURRENT_DIR = os.getcwd()
MODEL_PATH = os.path.join(CURRENT_DIR, 'best_student_model.joblib')
CSV_PATH = os.path.join(CURRENT_DIR, 'StudentPerformanceFactors.csv')

@st.cache_resource
def get_model():
    if not os.path.exists(MODEL_PATH):
        if not os.path.exists(CSV_PATH):
            st.error("Dataset `StudentPerformanceFactors.csv` not found!")
            st.stop()
        df = pd.read_csv(CSV_PATH)
        X = df.drop('Exam_Score', axis=1)
        y = df['Exam_Score']
        num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
        cat_cols = X.select_dtypes(include=['object']).columns.tolist()
        preprocessor = ColumnTransformer(transformers=[
            ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols),
            ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols)
        ])
        pipe = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', LinearRegression())])
        X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
        pipe.fit(X_tr, y_tr)
        joblib.dump(pipe, MODEL_PATH)
    return joblib.load(MODEL_PATH)

model = get_model()

# App Header
st.markdown('<p class="main-header">🎓 Student Performance Predictor</p>', unsafe_allow_html=True)
st.markdown('<p class="sub-header">Interactive Machine Learning Web Application for Predicting Academic Outcomes</p>', unsafe_allow_html=True)
st.markdown("---")

# Sidebar Configuration Form
st.sidebar.markdown("### ⚙️ Parameter Configuration")
st.sidebar.markdown("Configure student attributes to simulate final exam score outcomes.")

with st.sidebar.form("prediction_form"):

    with st.expander("📚 Study Habits & Attendance", expanded=True):
        hours_studied = st.slider("Hours Studied (per week)", 1, 50, 15)
        attendance = st.slider("Attendance Rate (%)", 0, 100, 85)
        previous_scores = st.slider("Previous Scores Average", 0, 100, 75)
        sleep_hours = st.slider("Sleep Hours (daily)", 4, 10, 7)
        tutoring_sessions = st.slider("Tutoring Sessions (monthly)", 0, 10, 2)
        physical_activity = st.slider("Physical Activity (hours/week)", 0, 10, 3)

    with st.expander("🏡 Support & Environment"):
        parental_involvement = st.selectbox("Parental Involvement", ["Low", "Medium", "High"], index=1)
        access_to_resources = st.selectbox("Access to Resources", ["Low", "Medium", "High"], index=1)
        motivation_level = st.selectbox("Motivation Level", ["Low", "Medium", "High"], index=1)
        family_income = st.selectbox("Family Income", ["Low", "Medium", "High"], index=1)
        internet_access = st.selectbox("Internet Access", ["Yes", "No"])
        extracurricular_activities = st.selectbox("Extracurricular Activities", ["Yes", "No"])

    with st.expander("🏫 School & Background"):
        teacher_quality = st.selectbox("Teacher Quality", ["Low", "Medium", "High"], index=1)
        peer_influence = st.selectbox("Peer Influence", ["Negative", "Neutral", "Positive"], index=2)
        parental_education_level = st.selectbox("Parental Education Level", ["High School", "College", "Postgraduate"], index=1)
        school_type = st.selectbox("School Type", ["Public", "Private"])
        distance_from_home = st.selectbox("Distance from Home", ["Near", "Moderate", "Far"])
        learning_disabilities = st.selectbox("Learning Disabilities", ["Yes", "No"])
        gender = st.selectbox("Gender", ["Male", "Female"])

    st.markdown("")
    submit_button = st.form_submit_button(label="🔮 Predict Exam Score", use_container_width=True)

# Main Dashboard Metric Highlights
m_col1, m_col2, m_col3 = st.columns(3)
with m_col1:
    st.metric(label="📖 Study Commitment", value=f"{hours_studied} hrs/week")
with m_col2:
    st.metric(label="📊 Attendance Metric", value=f"{attendance}%")
with m_col3:
    st.metric(label="📈 Academic Baseline", value=f"{previous_scores} / 100")

st.markdown("---")

# Prediction Execution Container
if submit_button:
    input_data = pd.DataFrame({
        'Hours_Studied': [hours_studied], 'Attendance': [attendance],
        'Parental_Involvement': [parental_involvement], 'Access_to_Resources': [access_to_resources],
        'Extracurricular_Activities': [extracurricular_activities], 'Sleep_Hours': [sleep_hours],
        'Previous_Scores': [previous_scores], 'Motivation_Level': [motivation_level],
        'Internet_Access': [internet_access], 'Tutoring_Sessions': [tutoring_sessions],
        'Family_Income': [family_income], 'Teacher_Quality': [teacher_quality],
        'School_Type': [school_type], 'Peer_Influence': [peer_influence],
        'Physical_Activity': [physical_activity], 'Learning_Disabilities': [learning_disabilities],
        'Parental_Education_Level': [parental_education_level], 'Distance_from_Home': [distance_from_home],
        'Gender': [gender]
    })

    prediction = model.predict(input_data)[0]

    st.markdown("### 🎯 Predictive Analytics Result")

    res_col1, res_col2 = st.columns([1, 2])
    with res_col1:
        st.metric(label="Estimated Final Score", value=f"{prediction:.2f} / 100")

    with res_col2:
        if prediction >= 80:
            st.success("🌟 **Outstanding Performance Outlook!** Current study discipline, attendance rates, and support indicators position this profile for top-tier academic standing.")
            st.balloons()
        elif prediction >= 60:
            st.info("👍 **Stable & Consistent Performance.** Current habits securely maintain a healthy academic standing within this grading bracket.")
        else:
            st.warning("⚠️ **Targeted Intervention Recommended.** Enhancing weekly study hours, tutoring frequency, or resource access can significantly elevate projected performance.")
else:
    st.info("👈 **Get Started:** Expand sections in the sidebar configuration form on the left, customize student metrics, and click **'Predict Exam Score'**.")

Overwriting app.py


In [2]:
# 1. Ensure all required packages are installed
!pip install -q streamlit pandas scikit-learn joblib numpy

import subprocess, time

# 2. Kill any stale processes
!pkill -f streamlit
!pkill -f cloudflared
time.sleep(2)

# 3. Start Streamlit silently in the background
subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port=8501", "--server.headless=true", "--logger.level=error"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(3)

# 4. Download cloudflared and launch the tunnel
!curl -sL --output cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared

print("🚀 Your live Cloudflare URL will appear below. Click the link ending in '.trycloudflare.com':")
!./cloudflared tunnel --url http://localhost:8501

🚀 Your live Cloudflare URL will appear below. Click the link ending in '.trycloudflare.com':
2026-10-26T21:15:42Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-10-26T21:15:42Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-10-26T21:15:45Z INF +--------------------------------------------------------------------------------------------+
2026-10-26T21:15:45Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be rea